# 🎯 Answer Accuracy Evaluation — Showcase Notebook

This notebook demonstrates a production-minded approach to evaluating **closed-book factual answer accuracy** across public benchmarks and custom golden datasets.

The purpose is not only to generate a score. The purpose is to show a disciplined evaluation workflow:

- choose appropriate datasets for the capability being tested
- normalize heterogeneous benchmark formats into one task schema
- make the target model explicit
- document metrics and their limitations
- run reproducible evaluations
- diagnose weak slices and suspicious cases
- produce artifacts that can support model comparison, governance review, and future regression testing

All heavy implementation lives in `src/genai_capability_bench`. This notebook is the guided interface and analysis layer.


---

## 🧭 1. Evaluation Framing

**Capability under test:** answer accuracy.

Answer accuracy asks whether a model can produce the expected answer to a factual question. It is deceptively simple: scoring a short answer like `Tokyo` is easy, but scoring a paraphrased legal, medical, financial, or literature answer can require stronger semantic matching or LLM-as-judge review.

This notebook starts with a pragmatic scoring stack:

| Evaluation Layer | Role |
|---|---|
| Deterministic lexical checks | Fast, cheap, reproducible baseline scoring |
| Lightweight semantic-ish similarity | Offline signal for wording variation |
| Review diagnostics | Surfaces low-score and metric-disagreement cases |
| Future extension points | Embeddings, LLM judges, multiple-choice scoring, domain rubrics |

**Important:** answer accuracy is not the same as truthfulness, reasoning, RAG faithfulness, or tool-use correctness. Those require related but distinct tests, and they belong in separate capability notebooks.


---

## 🗃️ 2. Dataset Strategy

This repo uses a reusable dataset registry. Public benchmarks and local/custom datasets are normalized into the same `EvalTask` schema so downstream evaluators can stay consistent.

There are two valid usage patterns:

1. **Public benchmark evaluation** — choose one or more registered public datasets, download from Hugging Face if needed, normalize, cache locally, then evaluate.
2. **Custom golden-set evaluation** — point to your own JSON/JSONL/CSV file with curated domain questions and reference answers.

For showcase or governance work, the dataset choice should be explicit. A good report should answer:

- Which dataset(s) were used?
- Which split(s) were used?
- How many examples per dataset?
- Which categories or subjects were included?
- Was the data downloaded fresh or loaded from local cache?
- Are the tasks closed-book, multiple-choice, context-grounded, or multi-hop?

The notebook can evaluate multiple datasets in one run through `DATASET_KEYS`.


---

## 📏 3. Metric Methodology

The current answer-accuracy evaluator uses four starter metrics and a composite score. These are intentionally transparent and cheap to run, which makes them useful for smoke tests and reproducible demos.

They are **not** the final word for production evaluation. For high-stakes or open-ended answer scoring, add embedding similarity, LLM-as-judge rubrics, or task-specific graders.


In [ ]:
# =============================================================================
# CELL 1: IMPORTS & NOTEBOOK SETUP
# =============================================================================
from pathlib import Path
import sys
import warnings

warnings.filterwarnings('ignore')

REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT / 'src'))

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

try:
    import ipywidgets as widgets
    WIDGETS_IMPORTABLE = True
except Exception:
    WIDGETS_IMPORTABLE = False

from genai_capability_bench.core.runner import load_config, parse_models
from genai_capability_bench.core.schemas import Capability
from genai_capability_bench.reporting.diagnostics import metric_guide_table, score_interpretation_table
from genai_capability_bench.reporting.notebook_views import dataset_catalog_table, model_config_table
from genai_capability_bench.reporting.plots import plot_capability_scores
from genai_capability_bench.workflows.answer_accuracy import (
    AnswerAccuracyRunConfig,
    load_answer_accuracy_tasks,
    run_answer_accuracy_workflow,
)

OUTPUT_ROOT = REPO_ROOT / 'outputs' / 'runs'

print('✅ Imports loaded')
print(f'📁 Repo root: {REPO_ROOT}')


In [ ]:
# =============================================================================
# CELL 2: DATASET CATALOG AND METRIC GUIDE
# =============================================================================
print('📚 Registered datasets')
display(dataset_catalog_table())

print('📏 Metrics used in the current answer-accuracy evaluator')
display(metric_guide_table())

print('🧮 Score interpretation bands')
display(score_interpretation_table())


---

## ⚙️ 4. Configuration

Edit the variables below before running.

### Dataset Selection

Use `DATASET_KEYS` to choose one or more datasets. Examples:

```python
DATASET_KEYS = ['answer_accuracy_sample']
DATASET_KEYS = ['mmlu', 'arc']
DATASET_KEYS = ['triviaqa', 'natural_questions']
DATASET_KEYS = ['custom']
```

For public datasets, install Hugging Face support first:

```bash
pip install -e ".[hf]"
```

Public datasets can be large. Start with a small `SAMPLE_SIZE_PER_DATASET`, confirm the workflow, then scale up.

### Target Model

Use the mock config for workflow validation. Use the OpenAI-compatible template for real model evaluation through `.env`.

```python
MODEL_CONFIG_PATH = REPO_ROOT / 'configs' / 'eval_core_demo.yaml'
# MODEL_CONFIG_PATH = REPO_ROOT / 'configs' / 'eval_openai_compatible_template.yaml'
```


In [ ]:
# =============================================================================
# CELL 3: CONFIGURATION BLOCK — edit this cell
# =============================================================================
USE_INTERACTIVE_WIDGETS = False  # Set True only if your frontend renders ipywidgets correctly.

MODEL_CONFIG_PATH = REPO_ROOT / 'configs' / 'eval_core_demo.yaml'
# MODEL_CONFIG_PATH = REPO_ROOT / 'configs' / 'eval_openai_compatible_template.yaml'

DATASET_KEYS = ['answer_accuracy_sample']
DATASET_SPLITS = {}  # Optional per-dataset overrides, e.g. {'mmlu': 'test', 'squad': 'validation'}
CUSTOM_DATASET_PATH = None

SAMPLE_SIZE_PER_DATASET = 10
SELECTED_CATEGORIES = 'ALL'  # 'ALL' or list, e.g. ['history', 'science']
RANDOM_SEED = 42
PASS_THRESHOLD = None  # None = read from model config default_pass_threshold

DOWNLOAD_IF_MISSING = True
CACHE_LOCAL_COPY = True
REFRESH_DATASET_CACHE = False
RUN_ID_PREFIX = 'answer_accuracy_demo'

if USE_INTERACTIVE_WIDGETS and WIDGETS_IMPORTABLE:
    dataset_options = dataset_catalog_table()['Dataset Key'].tolist()
    dataset_widget = widgets.SelectMultiple(
        options=dataset_options,
        value=tuple(DATASET_KEYS),
        description='Datasets',
        style={'description_width': 'initial'},
    )
    sample_widget = widgets.IntText(value=SAMPLE_SIZE_PER_DATASET, description='Sample / dataset', style={'description_width': 'initial'})
    display(widgets.VBox([dataset_widget, sample_widget]))
    print('Widget mode enabled. Adjust widgets, then re-run this cell and the cells below.')
    DATASET_KEYS = list(dataset_widget.value)
    SAMPLE_SIZE_PER_DATASET = int(sample_widget.value)
elif USE_INTERACTIVE_WIDGETS and not WIDGETS_IMPORTABLE:
    print('ipywidgets is not installed. Falling back to manual variables.')

workflow_config = AnswerAccuracyRunConfig(
    repo_root=REPO_ROOT,
    model_config_path=MODEL_CONFIG_PATH,
    dataset_keys=DATASET_KEYS,
    dataset_splits=DATASET_SPLITS,
    sample_size_per_dataset=SAMPLE_SIZE_PER_DATASET,
    selected_categories=SELECTED_CATEGORIES,
    custom_dataset_path=CUSTOM_DATASET_PATH,
    random_seed=RANDOM_SEED,
    pass_threshold=PASS_THRESHOLD,
    run_id_prefix=RUN_ID_PREFIX,
    output_root=OUTPUT_ROOT,
    download_if_missing=DOWNLOAD_IF_MISSING,
    cache_local_copy=CACHE_LOCAL_COPY,
    refresh_dataset_cache=REFRESH_DATASET_CACHE,
)

model_config = load_config(MODEL_CONFIG_PATH)
models = parse_models(model_config)

print('✅ Configuration loaded')
print(f'🎯 Dataset keys: {DATASET_KEYS}')
print(f'🤖 Target model(s): {[m.name for m in models]}')
display(model_config_table(models))


---

## ✅ 5. Pre-Flight Check

This cell materializes the selected datasets, normalizes them into `EvalTask`, and shows the exact evaluation scope before any model calls are made.

This is where a high-quality evaluation workflow should catch ambiguity early:

- no selected datasets
- unsupported dataset key
- missing custom dataset path
- no answer-accuracy tasks after filtering
- unexpected categories or tiny sample sizes


In [ ]:
# =============================================================================
# CELL 4: PRE-FLIGHT CHECK
# =============================================================================
tasks, dataset_manifest_df = load_answer_accuracy_tasks(workflow_config)

if not tasks:
    raise RuntimeError('❌ No answer-accuracy tasks selected. Check DATASET_KEYS, split, and category filters.')

preflight_df = pd.DataFrame([
    {'Check': 'Model config exists', 'Status': MODEL_CONFIG_PATH.exists(), 'Detail': str(MODEL_CONFIG_PATH)},
    {'Check': 'At least one model configured', 'Status': len(models) > 0, 'Detail': ', '.join(m.name for m in models)},
    {'Check': 'At least one dataset selected', 'Status': len(DATASET_KEYS) > 0, 'Detail': ', '.join(DATASET_KEYS)},
    {'Check': 'Answer-accuracy tasks available', 'Status': len(tasks) > 0, 'Detail': f'{len(tasks)} tasks'},
    {'Check': 'Output directory available', 'Status': OUTPUT_ROOT.exists() or OUTPUT_ROOT.parent.exists(), 'Detail': str(OUTPUT_ROOT)},
])

display(preflight_df)
print('📦 Dataset manifest')
display(dataset_manifest_df)

category_df = pd.DataFrame({'category': [t.category for t in tasks]}).value_counts().reset_index(name='n')
print('🏷️ Selected category distribution')
display(category_df)

if not all(preflight_df['Status']):
    raise RuntimeError('❌ Pre-flight failed. Fix failed checks before continuing.')

print('✅ Pre-flight passed')


---

## 🚀 6. Run Evaluation

The workflow below handles:

- client initialization
- per-model evaluation
- progress display
- normalized result generation
- summary and leaderboard tables
- diagnostic tagging
- artifact writing

For real LLMs, this is where API calls and cost begin. Keep `SAMPLE_SIZE_PER_DATASET` small until the run path is confirmed.


In [ ]:
# =============================================================================
# CELL 5: RUN ANSWER-ACCURACY WORKFLOW
# =============================================================================
run = run_answer_accuracy_workflow(workflow_config, show_progress=True)

print('✅ Evaluation complete')
print(f'📁 Run directory: {run.run_dir}')
print(f'📄 Report: {run.report_path}')

display(run.results_df[['model_name', 'task_id', 'category', 'actual_output', 'expected_output', 'score', 'passed']])


---

## 🔎 7. Result Analysis and Diagnostics

The point of diagnostics is to avoid over-trusting a single aggregate score.

Review especially:

- **High review priority**: likely wrong or very weak match
- **Near threshold**: borderline case that may flip under a better metric
- **Metric disagreement**: one metric says strong match while another says weak match
- **Category weakness**: model performs unevenly across subjects/domains

For portfolio-quality evaluation, diagnostics are as important as the final score.


In [ ]:
# =============================================================================
# CELL 6: SUMMARY, LEADERBOARD, AND DIAGNOSTICS
# =============================================================================
print('📋 Summary by model / capability / category')
display(run.summary_df)

print('🗃️ Dataset/category summary')
display(run.dataset_summary_df)

print('🏁 Capability leaderboard')
display(run.leaderboard_df)

print('🩺 Diagnostic review table')
review_cols = [
    'review_priority', 'model_name', 'task_id', 'category', 'actual_output',
    'expected_output', 'score', 'exact_match', 'contains_match', 'token_f1',
    'tfidf_similarity', 'metric_disagreement'
]
display(run.diagnostics_df.sort_values(['score', 'task_id'])[review_cols])


---

## 📊 8. Visualization

Visuals should communicate the evaluation story quickly:

- category score profile
- score distribution
- pass/fail balance
- review-priority mix

As datasets grow, these views become more informative than raw tables.


In [ ]:
# =============================================================================
# CELL 7: VISUALIZATION
# =============================================================================
plot_capability_scores(run.summary_df, title='Answer Accuracy by Category');

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(run.results_df['score'], bins=10, color='#2563eb', alpha=0.85, edgecolor='white')
threshold = workflow_config.pass_threshold or float(model_config.get('default_pass_threshold', 0.7))
axes[0].axvline(threshold, color='#dc2626', linestyle='--', label=f'Pass threshold ({threshold:.2f})')
axes[0].set_title('Score Distribution')
axes[0].set_xlabel('Score')
axes[0].set_ylabel('Responses')
axes[0].set_xlim(0, 1)
axes[0].grid(axis='y', alpha=0.25)
axes[0].legend()

priority_counts = run.diagnostics_df['review_priority'].value_counts()
axes[1].bar(priority_counts.index, priority_counts.values, color='#0f766e')
axes[1].set_title('Review Priority Mix')
axes[1].set_ylabel('Cases')
axes[1].tick_params(axis='x', rotation=30)
axes[1].grid(axis='y', alpha=0.25)

fig.tight_layout()
plt.show()


---

## 📝 9. Executive Summary and Artifacts

A credible evaluation report should preserve enough context for another person to understand and reproduce the run:

- target model(s)
- dataset(s), split(s), sample size, and cache/source path
- metric methodology
- aggregate scores
- diagnostics and review guidance
- raw outputs

The generated report is intentionally deterministic. Later, optional LLM-generated narrative can be added, but it should be clearly labeled as AI-generated commentary.


In [ ]:
# =============================================================================
# CELL 8: EXECUTIVE SUMMARY AND ARTIFACTS
# =============================================================================
display(Markdown('### Executive Summary'))
display(Markdown(run.summary_text))

print('📦 Saved artifacts')
for path in sorted(run.run_dir.iterdir()):
    print(f'- {path.name}')

print(f'
Report path: {run.report_path}')


---

## 🧠 10. Interpretation and Next Steps

A strong answer-accuracy workflow should evolve along several dimensions:

1. **Dataset quality**: add curated enterprise/domain golden sets, not just public benchmarks.
2. **Metric maturity**: add embedding similarity and LLM-as-judge scoring for open-ended answers.
3. **Task-specific grading**: use exact multiple-choice accuracy for MMLU/ARC and separate closed-book from context-grounded QA.
4. **Benchmark hygiene**: track splits, sample seeds, dataset versions, and cache paths.
5. **Model comparison**: run multiple models under the same datasets and thresholds.
6. **Regression testing**: compare new runs against previous baselines.

This notebook is the answer-accuracy slice of a broader GenAI capability suite. The same architecture should extend to truthfulness, instruction following, reasoning, RAG faithfulness, tool use, and agentic task completion.
